In [2]:
import pandas as pd
import sqlite3
import numpy as np
import mygene

In [3]:
# 1. Load the data from the csv file
#df = pd.read_csv("C:/amish/pancrea_proteomics/analysis/data/pancreas_proteomics.csv", header=1)
df = pd.read_csv("../data/pancreas_proteomics.csv", header=1)
#print("Read file")
#print(df.head)

In [4]:
# Annotate non-matrisome 
non_matrisome_annotation = pd.read_csv("../data/pantherGeneList.txt", sep="\t", names=['Gene ID', 'Annotated Gene', 'Gene Name', 'Annotated Matrisome Category'])
#non_matrisome.head()
df['Annotated Matrisome Category'] = df['Annotated Gene'].map(non_matrisome_annotation.set_index('Annotated Gene')['Annotated Matrisome Category']).fillna(df['Annotated Matrisome Category']).fillna("Unclassified")
df['Annotated Matrisome Category']

0                                       Collagens
1                                       Collagens
2                                       Collagens
3                                       Collagens
4                                       Collagens
                          ...                    
7416    homeodomain transcription factor(PC00119)
7417                                Non-matrisome
7418            scaffold/adaptor protein(PC00226)
7419                       phospholipase(PC00186)
7420                                 Unclassified
Name: Annotated Matrisome Category, Length: 7421, dtype: str

In [5]:
# Add annotation from GO Cellular component
mg = mygene.MyGeneInfo()
go_cc_ann = mg.querymany(df['Annotated Gene'].to_list(), scopes='uniprot', fields='go', species='human', as_dataframe=True)
go_cc_ann = go_cc_ann.reset_index()
go_cc_ann_clean = go_cc_ann.drop_duplicates(subset='query', keep='first')
go_cc_ann_clean = go_cc_ann_clean.reset_index(drop=True)
#results

29 input query terms found dup hits:	[('B7ZAQ6', 2), ('O60449', 2), ('P0C0L4', 2), ('P0C0L5', 2), ('P0C0S8', 5), ('P0DMM9', 2), ('P0DMV8'
75 input query terms found no hit:	['A0A075B6K5', 'A0A075B6R9', 'A0A075B6S5', 'A0A0A0MS15', 'A0A0B4J1U3', 'A0A0B4J1U7', 'A0A0B4J1V2', '


In [6]:
# Categorization function using GO Cellular Component
def categorize_go_cc(cc_data):
    # Check for NaN or None
    if isinstance(cc_data, float) or cc_data is None:
        return 'Unclassified'
    # Check for empty list
    if isinstance(cc_data, list) and len(cc_data)==0:
        return 'Unclassified'
    # Single dictionaries
    if isinstance(cc_data, dict):
        cc_data = [cc_data]
    
    term_names = " ".join([item.get('term', '').lower() for item in cc_data if isinstance(item, dict)])

    if 'mitochondri' in term_names:
        return 'Mitochondrial Proteins'
    elif 'nucleus' in term_names or 'nuclear' in term_names:
        return 'Nuclear Proteins'
    elif 'endoplasmic reticulum' in term_names or 'golgi' in term_names:
        return 'ER/Golgi Proteins'
    elif 'cytoskelet' in term_names or 'actin' in term_names or 'microtubule' in term_names:
        return 'Cytoskeletal Proteins'
    elif 'membrane' in term_names:
        return 'Membrane Proteins'
    elif 'cytoplasm' in term_names or 'cytosol' in term_names:
        return 'Cytosolic Proteins'
    elif 'secretory granule' in term_names or 'vesicle' in term_names or 'exosome' in term_names:
        return 'Secretory/Vesicular Proteins'
    elif 'extracellular' in term_names or 'secretes' in term_names:
        return 'ECM/secreted proteins'
    elif 'lysosom' in term_names or 'endosom' in term_names or 'autophagosom' in term_names:
        return 'Lysosomal/Endosomal Proteins'
    elif 'ribosom' in term_names or 'proteasom' in term_names:
        return 'Ribosomal/Proteasomal Proteins'
    elif 'plasma membrane' in term_names or 'cell surface' in term_names:
        return 'Plasma membrane Proteins'
    else:
        return 'Other'
    
# Apply the categorization function
go_cc_ann_clean['GO:CC'] = go_cc_ann_clean['go.CC'].apply(categorize_go_cc)


    
#len(results_clean)
#len(go_cc_ann_clean), go_cc_ann_clean.index


In [14]:
go_cc_ann_clean[['query', 'GO:CC']], df['Protein ID']

(       query                  GO:CC
 0     A6NMZ7  ECM/secreted proteins
 1     P02452      ER/Golgi Proteins
 2     P02458      ER/Golgi Proteins
 3     P02461      ER/Golgi Proteins
 4     P02462      ER/Golgi Proteins
 ...      ...                    ...
 7416  Q9Y6X8       Nuclear Proteins
 7417  Q9Y6X9       Nuclear Proteins
 7418  Q9Y6Y0       Nuclear Proteins
 7419  Q9Y6Y8      ER/Golgi Proteins
 7420     nan           Unclassified
 
 [7421 rows x 2 columns],
 0           A6NMZ7
 1           P02452
 2           P02458
 3           P02461
 4           P02462
            ...    
 7416        Q9Y6X8
 7417        Q9Y6X9
 7418        Q9Y6Y0
 7419        Q9Y6Y8
 7420    A0A0B4J1V0
 Name: Protein ID, Length: 7421, dtype: str)

In [15]:
# 2. Extract all annotations from the data
annotations = df[['Protein ID', 'Gene', 'Annotated Gene', 'Annotated Matrisome Division', 'Annotated Matrisome Category']].copy()
annotations['GO CC'] = go_cc_ann_clean['GO:CC'].values
print(annotations.tail())

      Protein ID      Gene Annotated Gene Annotated Matrisome Division  \
7416      Q9Y6X8      ZHX2         Q9Y6X8                Non-matrisome   
7417      Q9Y6X9     MORC2         Q9Y6X9                Non-matrisome   
7418      Q9Y6Y0  IVNS1ABP         Q9Y6Y0                Non-matrisome   
7419      Q9Y6Y8   SEC23IP         Q9Y6Y8                Non-matrisome   
7420  A0A0B4J1V0  IGHV3-15            NaN                          NaN   

                   Annotated Matrisome Category              GO CC  
7416  homeodomain transcription factor(PC00119)   Nuclear Proteins  
7417                              Non-matrisome   Nuclear Proteins  
7418          scaffold/adaptor protein(PC00226)   Nuclear Proteins  
7419                     phospholipase(PC00186)  ER/Golgi Proteins  
7420                               Unclassified       Unclassified  


In [16]:
intensity_cols = [col for col in df.columns if 'MaxLFQ' in col]
expression = df[['Protein ID'] + intensity_cols].copy()
expression_long = expression.melt(id_vars=['Protein ID'], var_name='Sample', value_name='Intensity')
print(expression_long)

        Protein ID                            Sample    Intensity
0           A6NMZ7  sample_01 MaxLFQ Total Intensity    153502.56
1           P02452  sample_01 MaxLFQ Total Intensity  98558584.00
2           P02458  sample_01 MaxLFQ Total Intensity   3466728.80
3           P02461  sample_01 MaxLFQ Total Intensity  70546624.00
4           P02462  sample_01 MaxLFQ Total Intensity  29273744.00
...            ...                               ...          ...
133573      Q9Y6X8  sample_20 MaxLFQ Total Intensity         0.00
133574      Q9Y6X9  sample_20 MaxLFQ Total Intensity     76899.05
133575      Q9Y6Y0  sample_20 MaxLFQ Total Intensity         0.00
133576      Q9Y6Y8  sample_20 MaxLFQ Total Intensity    364424.30
133577  A0A0B4J1V0  sample_20 MaxLFQ Total Intensity         0.00

[133578 rows x 3 columns]


In [17]:
expression_long['Intensity'] = expression_long['Intensity'].replace(0, np.nan)
expression_long['Sample ID'] = expression_long['Sample'].str.extract(r'(sample_\d+)')
expression_long['Condition'] = np.where(expression_long['Sample ID'].isin([f"sample_{i:02d}" for i in [1, 2, 3, 5, 6, 7, 8, 9, 10]]), 'T1D', 'Control')
print(expression_long)

        Protein ID                            Sample    Intensity  Sample ID  \
0           A6NMZ7  sample_01 MaxLFQ Total Intensity    153502.56  sample_01   
1           P02452  sample_01 MaxLFQ Total Intensity  98558584.00  sample_01   
2           P02458  sample_01 MaxLFQ Total Intensity   3466728.80  sample_01   
3           P02461  sample_01 MaxLFQ Total Intensity  70546624.00  sample_01   
4           P02462  sample_01 MaxLFQ Total Intensity  29273744.00  sample_01   
...            ...                               ...          ...        ...   
133573      Q9Y6X8  sample_20 MaxLFQ Total Intensity          NaN  sample_20   
133574      Q9Y6X9  sample_20 MaxLFQ Total Intensity     76899.05  sample_20   
133575      Q9Y6Y0  sample_20 MaxLFQ Total Intensity          NaN  sample_20   
133576      Q9Y6Y8  sample_20 MaxLFQ Total Intensity    364424.30  sample_20   
133577  A0A0B4J1V0  sample_20 MaxLFQ Total Intensity          NaN  sample_20   

       Condition  
0            T1D  
1

In [18]:
conn = sqlite3.connect("../data/pancreas_proteomics.db")
annotations.to_sql('annotations', conn, if_exists='replace', index=False)
expression_long[['Protein ID', 'Sample ID', 'Condition', 'Intensity']].to_sql('expression', conn, if_exists='replace', index=False) 

133578